# SLIDE Workflow

This notebook demonstrates sequence-free ruggedness inference on either a synthetic NK landscape or a complete empirical fitness landscape. For a population starting at genotype $\nu$, replicate-averaged mean fitness defines $F_\mu(\nu)$. The global squared-fitness decay is

$$G_\mu = \left\langle F_\mu(\nu)^2 \right\rangle_\nu.
$$

The workflow fits one local $\rho_1^{\mathrm{fit}}$ to every starting-genotype $F_\mu$ curve and one global $\rho_2^{\mathrm{fit}}$ to $G_\mu$.

## Setup

Load the existing SLIDE landscape, mutation-only diffusion, decay-fitting, and plotting functions. Run the notebook from the repository root so the package and empirical landscape paths resolve correctly.

In [ ]:
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np

from slide.data_generation import (
    all_start_locs,
    generate_empirical_decay_curves,
    load_empirical_landscape,
    nk_uniform_start_locs,
    run_nk_start_averaged_diffusion,
    uniform_start_locs,
)
from slide.direvo_functions import get_single_decay_rate, model_function

## Parameters

Choose `"nk"` for a reproducible binary NK landscape or `"empirical"` for one of the empirical landscapes supported by `load_empirical_landscape`. `NUM_STARTING_POINTS=None` uses every genotype; this is inexpensive for the default 1,024-genotype NK landscape but can be costly for a large empirical landscape. `TOTAL_MUTATION_RATE` is the expected mutations per sequence per generation and is divided by the number of sites before simulation.

In [ ]:
LANDSCAPE_SOURCE = "nk"  # "nk" or "empirical"
NUM_STARTING_POINTS: int | None = 20

NK_N = 10
NK_K = 3
NK_A = 2
EMPIRICAL_NAME = "GB1"

POPULATION_SIZE = 100
NUM_POPULATION_REPLICATES = 10
NUM_GENERATIONS = 40
TOTAL_MUTATION_RATE = 0.1
RANDOM_SEED = 42

## Generate $F_\mu$ and $G_\mu$

Select starting genotypes without replacement and run mutation-only population replicates from each one. For the NK branch, all starts share the same seeded landscape. For the empirical branch, fitness is read from the selected complete landscape. Averaging population replicates at fixed start estimates $F_\mu(\nu)$; squaring these curves before averaging over starts produces $G_\mu$.

In [ ]:
if LANDSCAPE_SOURCE not in {"nk", "empirical"}:
    raise ValueError("LANDSCAPE_SOURCE must be 'nk' or 'empirical'.")
if NUM_STARTING_POINTS is not None and NUM_STARTING_POINTS <= 0:
    raise ValueError("NUM_STARTING_POINTS must be positive or None.")

if LANDSCAPE_SOURCE == "nk":
    num_sites = NK_N
    num_alleles = NK_A
    total_genotypes = num_alleles**num_sites
    actual_num_starts = total_genotypes if NUM_STARTING_POINTS is None else NUM_STARTING_POINTS
    if actual_num_starts > total_genotypes:
        raise ValueError(f"Requested {actual_num_starts} starts from {total_genotypes} NK genotypes.")
    start_locations = nk_uniform_start_locs(
        n_sites=num_sites,
        num_alleles=num_alleles,
        num_starts=actual_num_starts,
    )
    replicate_curves = run_nk_start_averaged_diffusion(
        rng_key=jr.PRNGKey(RANDOM_SEED),
        trajectory_rng_key=jr.PRNGKey(RANDOM_SEED + 1),
        n_sites=num_sites,
        k=NK_K,
        num_alleles=num_alleles,
        starts=start_locations,
        popsize=POPULATION_SIZE,
        mutation_rate_per_site=TOTAL_MUTATION_RATE / num_sites,
        num_reps_per_start=NUM_POPULATION_REPLICATES,
        num_steps=NUM_GENERATIONS,
        return_replicates=True,
    )
    landscape_description = f"NK (N={NK_N}, K={NK_K}, A={NK_A})"
else:
    landscape = load_empirical_landscape(EMPIRICAL_NAME)
    num_sites = landscape.ndim
    num_alleles = landscape.shape[0]
    total_genotypes = landscape.size
    if NUM_STARTING_POINTS is None:
        start_locations = all_start_locs(landscape)
    else:
        if NUM_STARTING_POINTS > total_genotypes:
            raise ValueError(
                f"Requested {NUM_STARTING_POINTS} starts from {total_genotypes} empirical genotypes."
            )
        start_locations = uniform_start_locs(
            landscape,
            num_starts=NUM_STARTING_POINTS,
            seed=RANDOM_SEED,
            replace=False,
        )
    actual_num_starts = len(start_locations)
    replicate_curves = generate_empirical_decay_curves(
        landscape,
        mutation_rate=TOTAL_MUTATION_RATE / num_sites,
        popsize=POPULATION_SIZE,
        starts=start_locations,
        num_reps=NUM_POPULATION_REPLICATES,
        num_steps=NUM_GENERATIONS,
        seed=RANDOM_SEED,
    )
    landscape_description = f"empirical {EMPIRICAL_NAME} (shape={landscape.shape})"

f_mu = np.asarray(replicate_curves, dtype=float).mean(axis=1)
g_mu = np.square(f_mu).mean(axis=0)

print(f"Landscape: {landscape_description}")
print(f"Starting genotypes: {actual_num_starts} / {total_genotypes}")
print(f"Replicate trajectories: {replicate_curves.shape}")
print(f"F_mu: {f_mu.shape}; G_mu: {g_mu.shape}")

## Estimate Local and Global Decay Rates

Fit every start-level $F_\mu(\nu)$ with mutation scale $\mu$ to obtain $\rho_1^{\mathrm{fit}}(\nu)$. Fit the normalised global $G_\mu$ with scale $2\mu$, because squared-fitness decay follows an exponential of the form $\exp(-2\mu\rho_2)$; the returned parameter is therefore $\rho_2^{\mathrm{fit}}$. Failed local fits remain `NaN` at the matching row of `start_locations`.

In [ ]:
rho_1_fit = np.full(actual_num_starts, np.nan, dtype=float)
for start_index, curve in enumerate(f_mu):
    if not np.isfinite(curve).all() or np.isclose(curve[0], 0.0):
        continue
    try:
        rho_1_fit[start_index] = get_single_decay_rate(
            curve / curve[0],
            mut=TOTAL_MUTATION_RATE,
            num_steps=NUM_GENERATIONS,
        )[0]
    except (RuntimeError, ValueError, FloatingPointError):
        continue

if not np.isfinite(g_mu).all() or np.isclose(g_mu[0], 0.0):
    raise ValueError("G_mu must be finite and have a non-zero initial value.")
g_mu_normalised = g_mu / g_mu[0]
rho_2_fit, g_mu_asymptote = get_single_decay_rate(
    g_mu_normalised,
    mut=2.0 * TOTAL_MUTATION_RATE,
    num_steps=NUM_GENERATIONS,
)
generations = np.arange(NUM_GENERATIONS)
g_mu_fitted = model_function(
    generations,
    rho_2_fit,
    g_mu_asymptote,
    mut=2.0 * TOTAL_MUTATION_RATE,
)

finite_local_fits = np.isfinite(rho_1_fit)
print(f"rho_2^fit: {rho_2_fit:.4f}")
print(
    f"Successful rho_1^fit estimates: {finite_local_fits.sum()} / {actual_num_starts}"
)
if finite_local_fits.any():
    print(f"Mean rho_1^fit: {rho_1_fit[finite_local_fits].mean():.4f}")

## Global Squared-Fitness Decay

Compare the measured normalised $G_\mu$ trajectory with the fitted single-rate squared-decay model.

In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 3.4), dpi=150)
ax.scatter(generations, g_mu_normalised, s=24, color="tab:blue", label=r"Measured $G_\mu$")
ax.plot(
    generations,
    g_mu_fitted,
    color="tab:orange",
    linewidth=2,
    label=rf"Fit ($\rho_2^{{\mathrm{{fit}}}}={rho_2_fit:.3f}$)",
)
ax.set_xlabel("Generation")
ax.set_ylabel(r"Normalised $G_\mu$")
ax.set_title("Global squared-fitness decay")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## Local Decay-Rate Distribution

Show the distribution of successful $\rho_1^{\mathrm{fit}}$ estimates across the sampled starting genotypes.

In [ ]:
if not finite_local_fits.any():
    raise RuntimeError("No local decay curves could be fitted.")

local_fit_values = rho_1_fit[finite_local_fits]
local_fit_mean = local_fit_values.mean()
fig, ax = plt.subplots(figsize=(5.0, 3.4), dpi=150)
ax.hist(local_fit_values, bins="auto", color="tab:blue", alpha=0.75, edgecolor="white")
ax.axvline(
    local_fit_mean,
    color="tab:orange",
    linewidth=2,
    label=rf"Mean $={local_fit_mean:.3f}$; $n={len(local_fit_values)}$",
)
ax.set_xlabel(r"$\rho_1^{\mathrm{fit}}$")
ax.set_ylabel("Starting-genotype count")
ax.set_title("Local fitted decay rates")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()